# 紅酒課程：資料切分、標準化、評估與多項式迴歸

本 Notebook 完成紅酒 Part 1 的先備：先用 EDA 檢查資料，再完成 scikit-learn 的基本建模流程與多項式迴歸。

## 1. 載入工具與紅酒資料

資料的每一列是一瓶紅酒；`quality` 是要預測的品質分數，其他欄位是模型可用的特徵。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 這個相對路徑以「預習實作.ipynb 所在的 L4預習 資料夾」為起點。
# 坑：Notebook 的目前工作目錄由 Jupyter 啟動位置決定，不一定是這份檔案所在位置。
# 若找不到檔案，先執行 print(Path.cwd())，再依實際工作目錄調整路徑。
data_path = Path("../附件/L4 課程範例檔/dataset/winequality-red.csv")

# 專案內這份 CSV 使用逗號分隔，pd.read_csv() 的預設值正好是逗號。
# 坑：若誤寫 sep=';'，整列會被讀成一個欄位；讀完應確認 shape 是 (1599, 12)。
wine = pd.read_csv(data_path)
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
print(wine.shape)
print(wine.columns)

(1599, 12)
Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='str')


初步檢查資料

In [8]:
# print(wine.shape)
# wine.info()
print(wine.isna().sum())
# wine.describe().T

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


In [9]:
# X 儲存所有輸入特徵；drop 不會改動原本的 wine DataFrame。
# y 儲存每一列對應的正確品質分數，後面會拿來訓練與評估。
X = wine.drop(columns="quality")
y = wine["quality"]

# 核心資料契約：X 必須是二維的 (樣本數, 特徵數)，y 是一維的 (樣本數,)。
print(f"X.shape: {X.shape}")
print(f"y.shape: {y.shape}")

X.shape: (1599, 11)
y.shape: (1599,)


## 2. EDA：確認資料能不能直接建模

EDA（探索式資料分析）在模型訓練前檢查欄位型別、範圍、缺失值、分布、欄位關係與異常值。它的用途是理解資料，不是立刻刪除看起來奇怪的資料。

In [10]:
# info() 顯示每欄的型別與非空值數量；本課的線性迴歸需要可轉成數值的特徵。
wine.info()

# describe() 顯示數量、平均數、標準差、最小值與四分位數。
# 注意量級差異：不同欄位量級差很多，正是後面要標準化的原因。
wine.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


,count,mean,std,min,25%,50%,75%,max
fixed acidity,1599.0,8.319637,1.741096,4.60000,7.1000,7.90000,9.200000,15.90000
volatile acidity,1599.0,0.527821,0.179060,0.12000,0.3900,0.52000,0.640000,1.58000
citric acid,1599.0,0.270976,0.194801,0.00000,0.0900,0.26000,0.420000,1.00000
residual sugar,1599.0,2.538806,1.409928,0.90000,1.9000,2.20000,2.600000,15.50000
chlorides,1599.0,0.087467,0.047065,0.01200,0.0700,0.07900,0.090000,0.61100
free sulfur dioxide,1599.0,15.874922,10.460157,1.00000,7.0000,14.00000,21.000000,72.00000
total sulfur dioxide,1599.0,46.467792,32.895324,6.00000,22.0000,38.00000,62.000000,289.00000
density,1599.0,0.996747,0.001887,0.99007,0.9956,0.99675,0.997835,1.00369
pH,1599.0,3.311113,0.154386,2.74000,3.2100,3.31000,3.400000,4.01000
sulphates,1599.0,0.658149,0.169507,0.33000,0.5500,0.62000,0.730000,2.00000


In [11]:
# isna() 對每格資料判斷是否缺失；sum() 將每欄 True 加總成缺失值數量。
missing_counts = wine.isna().sum()
print("每欄缺失值數量：")
print(missing_counts)
print(f"完全重複的資料列數量：{wine.duplicated().sum()}")

# 坑：有缺失值時，不要直接 dropna()。先確認缺失比例和欄位意義，再決定刪除或填補。

每欄缺失值數量：
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64
完全重複的資料列數量：240


In [12]:
# value_counts() 顯示每個品質分數出現幾次，讓我們確認答案是否集中在少數分數。
# 坑：分布不平均不代表資料一定錯，但代表少數分數的預測通常較不可靠。
print(wine["quality"].value_counts().sort_index())

# corr() 是線性相關係數；接近 1 或 -1 代表關係較強，接近 0 代表沒有明顯線性關係。
# 坑：相關不等於因果。它不能單獨證明某個欄位造成品質改變。
print(wine.corr(numeric_only=True)["quality"].sort_values(ascending=False))

quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64
quality                 1.000000
alcohol                 0.476166
sulphates               0.251397
citric acid             0.226373
fixed acidity           0.124052
residual sugar          0.013732
free sulfur dioxide    -0.050656
pH                     -0.057731
chlorides              -0.128907
density                -0.174919
total sulfur dioxide   -0.185100
volatile acidity       -0.390558
Name: quality, dtype: float64


In [13]:
# IQR 異常值檢查：q1、q3 分別是第 25、75 百分位數；iqr 是兩者差距。
# outlier_rows 表示一列資料是否至少有一個特徵超出 q1 - 1.5×IQR 到 q3 + 1.5×IQR 的範圍。
feature_columns = X.columns
q1 = wine[feature_columns].quantile(0.25)
q3 = wine[feature_columns].quantile(0.75)
iqr = q3 - q1
outlier_rows = (
    (wine[feature_columns] < q1 - 1.5 * iqr) | (wine[feature_columns] > q3 + 1.5 * iqr)
).any(axis=1)
print(f"至少含一個 IQR 異常值的資料列：{outlier_rows.sum()} / {len(wine)}")

# 坑：IQR 只標記『值得檢查』的候選值。它可能是合理但罕見的紅酒，不能直接當成錯誤資料刪除。

至少含一個 IQR 異常值的資料列：405 / 1599


## 3. 切分資料與標準化

`train_test_split` 會隨機把資料分成訓練集與測試集。模型只能從訓練集學習；測試集保留到最後，模擬它從未看過的新紅酒。

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# test_size=0.2：20% 的資料留作測試。random_state 固定後，每次切分結果相同，方便重現。
print(f"訓練特徵：{X_train.shape}；測試特徵：{X_test.shape}")

scaler = StandardScaler()
# fit_transform：由 X_train 計算每個欄位的平均數與標準差，再轉換 X_train。
X_train_scaled = scaler.fit_transform(X_train)
# 坑：絕對不要對 X_test 使用 fit_transform。那會讓測試資料自己參與計算轉換規則，造成資料洩漏。
X_test_scaled = scaler.transform(X_test)

訓練特徵：(1279, 11)；測試特徵：(320, 11)


## 4. 線性迴歸基準模型與 MSE／R²

先建立最簡單的基準模型。MSE 越小越好；R² 越接近 1 越好，0 表示模型大致不比直接猜平均品質好。

In [15]:
baseline_model = LinearRegression()
# baseline_model 會從 X_train_scaled 與 y_train 學習各特徵的線性係數。
baseline_model.fit(X_train_scaled, y_train)

# coef_ 中每個值對應一個標準化後的特徵係數；正值代表該特徵增加時預測品質傾向增加。
# 坑：係數描述模型在這份資料中的關聯，不足以單獨證明因果。
print(pd.Series(baseline_model.coef_, index=X.columns).sort_values())

# y_pred 是模型對每筆測試紅酒預測出的品質分數。
y_pred = baseline_model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"線性迴歸 MSE: {mse:.4f}")
print(f"線性迴歸 R²: {r2:.4f}")

# 坑：兩個指標的第一個參數一定是實際答案 y_test，第二個才是預測值 y_pred。

volatile acidity       -0.179439
total sulfur dioxide   -0.120008
chlorides              -0.089084
pH                     -0.060610
citric acid            -0.027512
density                -0.019204
residual sugar          0.009421
fixed acidity           0.039789
free sulfur dioxide     0.058015
sulphates               0.146815
alcohol                 0.296628
dtype: float64
線性迴歸 MSE: 0.3900
線性迴歸 R²: 0.4032


## 5. 多項式迴歸

多項式迴歸不是另一種模型；它先用 `PolynomialFeatures` 增加平方項與特徵交互項，再交給 `LinearRegression`。例如原本的 `a`、`b`，二次轉換後可能有 `a²`、`a × b`、`b²`。

In [16]:
# degree=2 代表加入二次項與兩兩交互項。include_bias=False 避免和 LinearRegression 的截距欄位重複。
poly = PolynomialFeatures(degree=2, include_bias=False)

# 先在訓練集 fit，再用完全相同的欄位規則轉換測試集。
# 即使 PolynomialFeatures 不會計算平均數，養成此順序仍可避免前處理流程混亂。
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"原始特徵數：{X_train.shape[1]}")
print(f"二次多項式特徵數：{X_train_poly.shape[1]}")

# 坑：特徵數增加很快。degree 設得過高，模型可能在訓練資料表現很好、在測試資料表現很差（過度擬合）。

原始特徵數：11
二次多項式特徵數：77


In [17]:
# 多項式欄位也要標準化；仍只從訓練集 fit，測試集只 transform，避免資料洩漏。
poly_scaler = StandardScaler()
X_train_poly_scaled = poly_scaler.fit_transform(X_train_poly)
X_test_poly_scaled = poly_scaler.transform(X_test_poly)

# poly_model 是線性迴歸模型；『多項式』來自前面的特徵轉換，而不是模型本身。
poly_model = LinearRegression()
poly_model.fit(X_train_poly_scaled, y_train)
y_poly_pred = poly_model.predict(X_test_poly_scaled)

poly_mse = mean_squared_error(y_test, y_poly_pred)
poly_r2 = r2_score(y_test, y_poly_pred)
print(f"二次多項式迴歸 MSE: {poly_mse:.4f}")
print(f"二次多項式迴歸 R²: {poly_r2:.4f}")

# 重點不是多項式模型一定比較好，而是只根據同一份測試集的 MSE 與 R² 比較兩個模型。

二次多項式迴歸 MSE: 0.3819
二次多項式迴歸 R²: 0.4157


## 執行後應能回答

- `X` 為什麼是二維、`y` 為什麼是一維？
- 為什麼 scaler 只可以對 `X_train` 做 `fit`？
- MSE 與 R² 分別怎麼判斷表現？
- `PolynomialFeatures` 為何讓 column 變多，以及為何 degree 太高可能過度擬合？